# v4 D8 river-segment delta-contribution analysis

This notebook delineates selected river segments' incremental D8 drainage areas in the shared S04/S10 900 m cell. It intentionally does not select hillslope start points.


In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore', module='pyproj')


In [ ]:
# Case and editable selection
import json
with open('config.json') as f:
    config = json.load(f)
case_config = config['case']
watershed_name, hucs, site_name = case_config['watershed_name'], [case_config['hucs']], case_config['site_name']
if site_name != 'S04S10':
    raise ValueError("This v4 notebook requires config.json site_name 'S04S10'.")

# First run with no pairs, inspect P01--P48, then add pairs such as ('P03', 'P11').
selected_endpoint_pairs = [
    ('P40', 'P48'),
]
MATCH_TOLERANCE_M = 10.0
STREAM_ACCUMULATION_THRESHOLD = 100
if len({tuple(sorted(pair)) for pair in selected_endpoint_pairs}) != len(selected_endpoint_pairs):
    raise ValueError('Each selected endpoint pair must be unique.')


In [ ]:
# Imports, watershed, rivers, and DayMet DEM
import os, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
import rasterio.transform
import rasterio.windows
from matplotlib import pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
from pyproj import Transformer
from pysheds.grid import Grid
from shapely.geometry import Point
import watershed_workflow
import watershed_workflow.crs
import watershed_workflow.source_list
import watershed_workflow.split_hucs

site_selection_dir = Path('./site_selection')
output_dir = site_selection_dir / 'd8_delta_contribution'
output_dir.mkdir(exist_ok=True)
crs = watershed_workflow.crs.daymet_crs()
sources = watershed_workflow.source_list.get_default_sources()
sources['hydrography'] = watershed_workflow.source_list.hydrography_sources['NHD Plus']
sources['HUC'] = watershed_workflow.source_list.huc_sources['NHD Plus']
sources['DEM'] = watershed_workflow.source_list.dem_sources['NED 1/3 arc-second']

_, huc_shapes = watershed_workflow.get_hucs(sources['HUC'], hucs[0], 12, crs)
watershed = watershed_workflow.split_hucs.SplitHUCs(huc_shapes)
_, reaches = watershed_workflow.get_reaches(sources['hydrography'], hucs[0], watershed.exterior(), crs, crs, in_network=True, properties=True)
rivers = sorted(watershed_workflow.construct_rivers(reaches, method='hydroseq', ignore_small_rivers=2, prune_by_area=0.0, remove_diversions=True, remove_braided_divergences=True), key=len)
dem_profile, dem = watershed_workflow.get_raster_on_shape(sources['DEM'], watershed.exterior(), crs, out_crs=crs)
os.makedirs('./data/dem', exist_ok=True)
dem_path = './data/dem/reprojected_dem.tif'
with rasterio.open(dem_path, 'w', **dem_profile) as dst:
    dst.write(dem, 1)
dem_bounds = rasterio.transform.array_bounds(dem.shape[0], dem.shape[1], dem_profile['transform'])


In [ ]:
# Load the s1-v4 endpoint contract and recreate the stable s2-v4 P labels
raw_points = pd.read_csv(site_selection_dir / 'configured_site_river_segment_endpoints.csv')
required = {'site_id','input_latitude','input_longitude','raster_row','raster_col','point_id','endpoint_type','easting','northing','longitude','latitude'}
missing = required.difference(raw_points.columns)
if missing:
    raise ValueError(f'Endpoint CSV is missing columns: {sorted(missing)}')
if len(raw_points) != 96:
    raise ValueError(f'Expected 96 source rows, found {len(raw_points)}.')
pairs = raw_points[['raster_row','raster_col']].drop_duplicates()
if len(pairs) != 1 or tuple(pairs.iloc[0].astype(int)) != (25, 23):
    raise ValueError(f'Expected shared raster cell (25, 23), found {pairs.to_dict("records")}.')
grid_row, grid_col = 25, 23
grid_id = 'cell_r025_c023'
endpoints = raw_points.groupby(['easting','northing'], sort=True, as_index=False).agg(endpoint_type=('endpoint_type','first'), longitude=('longitude','first'), latitude=('latitude','first'), source_site_ids=('site_id',lambda x: ';'.join(sorted(set(x)))))
if len(endpoints) != 48:
    raise ValueError(f'Expected 48 physical endpoints, found {len(endpoints)}.')
endpoints.insert(0, 'point_label', [f'P{i:02d}' for i in range(1,49)])
site_locations = raw_points[['site_id','input_latitude','input_longitude']].drop_duplicates('site_id')
if set(site_locations.site_id) != {'S04','S10'}:
    raise ValueError('Endpoint CSV must contain both S04 and S10.')
with open(site_selection_dir / 'river_cells_and_segments.pkl', 'rb') as f:
    river_data = pickle.load(f)
line_segments, cache_crs = river_data['line_segments_in_cells'], river_data['crs']
to_daymet = Transformer.from_crs('EPSG:4326', cache_crs, always_xy=True)
endpoints['x_daymet'], endpoints['y_daymet'] = to_daymet.transform(endpoints.longitude.to_numpy(), endpoints.latitude.to_numpy())
site_locations['x_daymet'], site_locations['y_daymet'] = to_daymet.transform(site_locations.input_longitude.to_numpy(), site_locations.input_latitude.to_numpy())
endpoint_lookup = endpoints.set_index('point_label')
print(f'Validated {len(raw_points)} rows -> {len(endpoints)} P-labeled endpoints in {grid_id}.')
display(endpoints[['point_label','endpoint_type','source_site_ids','longitude','latitude']])


In [ ]:
# Resolve endpoint-label pairs to one cached river segment
def endpoint_distances(segment, point):
    coords = list(segment.coords)
    return Point(coords[0]).distance(Point(point)), Point(coords[-1]).distance(Point(point))

def resolve_segment(label_a, label_b):
    if label_a == label_b:
        raise ValueError(f'{label_a}: select two different endpoint labels.')
    unknown = [label for label in (label_a,label_b) if label not in endpoint_lookup.index]
    if unknown:
        raise ValueError(f'Unknown endpoint label(s) {unknown}; use P01--P48.')
    a, b = endpoint_lookup.loc[label_a], endpoint_lookup.loc[label_b]
    candidates = []
    for segment_id, segment in enumerate(line_segments, start=1):
        a_start, a_end = endpoint_distances(segment, (a.x_daymet,a.y_daymet))
        b_start, b_end = endpoint_distances(segment, (b.x_daymet,b.y_daymet))
        for da, db in ((a_start,b_end),(a_end,b_start)):
            if da <= MATCH_TOLERANCE_M and db <= MATCH_TOLERANCE_M:
                candidates.append((segment_id,segment,da,db))
    if not candidates:
        raise ValueError(f'{label_a}/{label_b} do not match the two endpoints of one cached segment within {MATCH_TOLERANCE_M:g} m.')
    if len(candidates) != 1:
        raise ValueError(f'{label_a}/{label_b} match multiple cached segments: {[x[0] for x in candidates]}.')
    segment_id, segment, da, db = candidates[0]
    return {'segment_id':segment_id,'segment':segment,'label_a':label_a,'label_b':label_b,'match_distance_a_m':da,'match_distance_b_m':db}
# Cached segments are a map overlay only.  Selected labels are resolved by D8 flow, not a 10 m segment-endpoint match.
selected_segments = []
print(f'Cached segments: {len(line_segments)}; selected endpoint pairs: {len(selected_endpoint_pairs)}')


In [ ]:
# Endpoint review visualization: all labels, selected pair(s), DEM, rivers, S04/S10, and the shared cell
with rasterio.open(site_selection_dir / 'burn_severity_900m_weighted.tif') as src:
    raster_crs = src.crs
    cell_bounds = rasterio.windows.bounds(rasterio.windows.Window(grid_col, grid_row, 1, 1), src.transform)
raster_to_daymet = Transformer.from_crs(raster_crs, cache_crs, always_xy=True)
left,bottom,right,top = cell_bounds
cell_corners = [raster_to_daymet.transform(x,y) for x,y in [(left,bottom),(right,bottom),(right,top),(left,top)]]
fig, ax = plt.subplots(figsize=(12,10))
image = ax.imshow(dem, cmap='terrain', extent=dem_bounds, origin='upper', zorder=0)
watershed_workflow.plot.rivers(rivers, cache_crs, ax=ax, colors='steelblue', linewidth=0.7, zorder=2)
for segment in line_segments:
    xy = np.asarray(segment.coords); ax.plot(xy[:,0],xy[:,1],color='gold',linewidth=1,alpha=.6,zorder=3)
ax.add_patch(MplPolygon(cell_corners,closed=True,fill=False,edgecolor='magenta',linewidth=2.5,linestyle='--',label=grid_id))
for point in endpoints.itertuples():
    ax.plot(point.x_daymet,point.y_daymet,'wo',markersize=4,markeredgecolor='black',zorder=5)
    ax.annotate(point.point_label,(point.x_daymet,point.y_daymet),xytext=(3,3),textcoords='offset points',fontsize=7,zorder=6)
for site in site_locations.itertuples():
    ax.plot(site.x_daymet,site.y_daymet,'k*',markersize=14,zorder=8,label=f'{site.site_id} input')
for chosen in selected_segments:
    xy=np.asarray(chosen['segment'].coords)
    ax.plot(xy[:,0],xy[:,1],color='crimson',linewidth=3,zorder=7,label=f"segment {chosen['segment_id']}: {chosen['label_a']}/{chosen['label_b']}")
margin=450; xs,ys=zip(*cell_corners)
ax.set(xlim=(min(xs)-margin,max(xs)+margin),ylim=(min(ys)-margin,max(ys)+margin),xlabel='DayMet easting [m]',ylabel='DayMet northing [m]',title='S04/S10 shared cell: endpoint labels and selected river segments')
ax.legend(loc='best',fontsize=9); ax.grid(True,alpha=.25)
plt.colorbar(image,ax=ax,fraction=.046,pad=.04,label='Elevation [m]')
plt.tight_layout(); plt.show()
if not selected_segments:
    print('No pairs selected. Inspect labels above, edit selected_endpoint_pairs, and rerun from the selection cell.')


In [ ]:
# Primary route-first workflow: P-label segment graph, route drainage, then delta area.
from collections import deque
from scipy import ndimage

if selected_endpoint_pairs:
    grid = Grid.from_raster(dem_path)
    dem_grid = grid.read_raster(dem_path)
    fdir = grid.flowdir(grid.resolve_flats(grid.fill_depressions(grid.fill_pits(dem_grid))), out_name='dir')
    accumulation = grid.accumulation(fdir)
    stream_mask = accumulation > STREAM_ACCUMULATION_THRESHOLD
    affine = grid.viewfinder.affine
    cell_area_m2 = abs(affine.a * affine.e - affine.b * affine.d)

    def cached_label(xy):
        labels = [row.point_label for row in endpoints.itertuples()
                  if Point((row.x_daymet, row.y_daymet)).distance(Point(xy)) <= 10.0]
        return labels[0] if len(labels) == 1 else None

    graph = {label: set() for label in endpoint_lookup.index}
    for segment in line_segments:
        first, last = cached_label(segment.coords[0]), cached_label(segment.coords[-1])
        if first and last and first != last:
            graph[first].add(last)
            graph[last].add(first)

    def find_route(first, last):
        queue, visited = deque([(first, [first])]), {first}
        while queue:
            node, route = queue.popleft()
            if node == last:
                return route
            for neighbor in sorted(graph[node]):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append((neighbor, route + [neighbor]))
        raise ValueError(f'No cached river-segment route connects {first} and {last}.')

    route_rows = []
    route_results = []
    for requested_a, requested_b in selected_endpoint_pairs:
        labels = find_route(requested_a, requested_b)
        print('River-segment route:', ' -> '.join(labels))
        print(f'Neighbors of {requested_a}:', '; '.join(sorted(graph[requested_a])) or '(none)')
        print(f'Neighbors of {requested_b}:', '; '.join(sorted(graph[requested_b])) or '(none)')

        masks = []
        for order, label in enumerate(labels, start=1):
            point = endpoint_lookup.loc[label]
            x_snap, y_snap = grid.snap_to_mask(stream_mask, (point.x_daymet, point.y_daymet))
            col, row = grid.nearest_cell(x_snap, y_snap)
            catchment = np.asarray(grid.catchment(x=col, y=row, fdir=fdir, xytype='index'), dtype=bool)
            edge = catchment & (ndimage.binary_erosion(catchment) != catchment)
            masks.append((label, catchment, edge, x_snap, y_snap))
            route_rows.append({'requested_endpoint_a': requested_a, 'requested_endpoint_b': requested_b,
                               'route_order': order, 'point_label': label,
                               'drainage_cells': int(catchment.sum()),
                               'drainage_area_m2': float(catchment.sum() * cell_area_m2)})

        upstream_i, downstream_i = (0, -1) if masks[0][1].sum() < masks[-1][1].sum() else (-1, 0)
        delta_mask = masks[downstream_i][1] & ~masks[upstream_i][1]
        for record in route_rows[-len(labels):]:
            record['upstream_endpoint_label'] = masks[upstream_i][0]
            record['downstream_endpoint_label'] = masks[downstream_i][0]
            record['delta_contribution_area_m2'] = float(delta_mask.sum() * cell_area_m2)

        route_results.append({'requested_a': requested_a, 'requested_b': requested_b,
                              'labels': labels, 'masks': masks, 'upstream_i': upstream_i,
                              'downstream_i': downstream_i, 'delta_mask': delta_mask})
        print(f'Catchment difference ({masks[downstream_i][0]} - {masks[upstream_i][0]}): {delta_mask.sum() * cell_area_m2:.0f} m²')

    route_csv = output_dir / 'river_segment_route_point_drainage_areas.csv'
    pd.DataFrame(route_rows).to_csv(route_csv, index=False)
    display(pd.DataFrame(route_rows))
    print(f'Saved {route_csv}')
else:
    print('No endpoint pairs selected; enter a pair such as ("P40", "P48").')


In [ ]:
# Route drainage visualization

if selected_endpoint_pairs:
    # array_bounds returns (west, south, east, north); imshow needs (left, right, bottom, top).
    dem_extent = (dem_bounds[0], dem_bounds[2], dem_bounds[1], dem_bounds[3])

    overview_points = endpoint_lookup.loc[['P40', 'P42', 'P44', 'P47', 'P48']]
    all_masks = [mask for result in route_results for mask in result['masks']]
    overview_boundaries = []
    for label, catchment, edge, x_snap, y_snap in all_masks:
        rows, cols = np.where(edge)
        x, y = affine * (cols, rows)
        overview_boundaries.append((label, x, y))

    overview_x = np.concatenate([x for label, x, y in overview_boundaries] + [np.asarray(xs), overview_points.x_daymet.to_numpy()])
    overview_y = np.concatenate([y for label, x, y in overview_boundaries] + [np.asarray(ys), overview_points.y_daymet.to_numpy()])
    overview_margin = max(500, 0.05 * max(np.ptp(overview_x), np.ptp(overview_y)))

    fig, ax = plt.subplots(figsize=(12, 10))
    image = ax.imshow(dem, cmap='terrain', extent=dem_extent, origin='upper', zorder=0)
    watershed_workflow.plot.rivers(rivers, cache_crs, ax=ax, colors='gold', linewidth=0.7, zorder=2)
    ax.plot([], [], color='gold', linewidth=0.7, label='NHD Plus river network')
    ax.add_patch(MplPolygon(cell_corners, closed=True, fill=False, edgecolor='magenta', linewidth=2.5, linestyle='--', label=grid_id))
    for site in site_locations.itertuples():
        ax.plot(site.x_daymet, site.y_daymet, 'k*', markersize=14, zorder=8, label=f'{site.site_id} input')
    for point in overview_points.itertuples():
        ax.plot(point.x_daymet, point.y_daymet, 'wo', markersize=5, markeredgecolor='black', zorder=7)
    for color, (label, x, y) in zip(plt.cm.tab20(np.linspace(0, 1, len(overview_boundaries))), overview_boundaries):
        ax.scatter(x, y, s=7, color=color, alpha=0.65, label=f'{label} drainage boundary', zorder=3)
    ax.set(xlim=(overview_x.min() - overview_margin, overview_x.max() + overview_margin),
           ylim=(overview_y.min() - overview_margin, overview_y.max() + overview_margin),
           xlabel='Easting [m]', ylabel='Northing [m]',
           title='Route drainage boundaries overview')
    ax.legend(loc='best', fontsize=8)
    plt.colorbar(image, ax=ax, fraction=.046, pad=.04, label='Elevation [m]')
    ax.grid(True, alpha=0.25)
    overview_path = output_dir / 'route_drainage_boundaries_overview.png'
    fig.savefig(overview_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved {overview_path}')

    for result in route_results:
        requested_a, requested_b = result['requested_a'], result['requested_b']
        labels, masks = result['labels'], result['masks']

        fig, ax = plt.subplots(figsize=(12, 10))
        watershed_workflow.plot.rivers(rivers, cache_crs, ax=ax, colors='gold', linewidth=0.7, zorder=2)
        ax.plot([], [], color='gold', linewidth=0.7, label='NHD Plus river network')
        ax.add_patch(MplPolygon(cell_corners, closed=True, fill=False, edgecolor='magenta', linewidth=2.5, linestyle='--', label=grid_id))
        for site in site_locations.itertuples():
            ax.plot(site.x_daymet, site.y_daymet, 'k*', markersize=14, zorder=8, label=f'{site.site_id} input')
        for color, (label, catchment, edge, x_snap, y_snap) in zip(plt.cm.tab10(np.linspace(0, 1, len(masks))), masks):
            rows, cols = np.where(edge)
            x, y = affine * (cols, rows)
            ax.scatter(x, y, s=7, color=color, alpha=0.65, label=f'{label} drainage boundary')
            ax.plot(x_snap, y_snap, 'o', color=color, markersize=8)
            ax.annotate(label, (x_snap, y_snap), xytext=(5, 5), textcoords='offset points', fontweight='bold', fontsize=7)
        ax.set(xlim=(min(xs) - margin, max(xs) + margin), ylim=(min(ys) - margin, max(ys) + margin),
               xlabel='Easting [m]', ylabel='Northing [m]',
               title=f'Route drainage boundaries: {" -> ".join(labels)}')
        ax.legend(loc='best', fontsize=8)
        ax.grid(True, alpha=0.25)
        figure_path = output_dir / f'{requested_a}_{requested_b}_route_drainage_boundaries.png'
        fig.savefig(figure_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved {figure_path}')
else:
    print('No endpoint pairs selected; enter a pair such as ("P40", "P48").')


In [ ]:
# Confirm hillslope selection before calculating start points

flag_hillslope_selection_confirmed = True

if not flag_hillslope_selection_confirmed:
    raise RuntimeError(
        'Inspect the route drainage plots above. When you have identified the hillslope to evaluate, '
        'set flag_hillslope_selection_confirmed = True'
    )


In [ ]:
# Calculate potential hillslope start points from two P labels

hillslope_endpoint_labels = ('P40', 'P44')  # Change these two labels after confirming the desired hillslope.
hillslope_site_id = 'S04'  # Site whose D8-snapped river point is the hillslope end point.

if not flag_hillslope_selection_confirmed:
    raise RuntimeError('Set flag_hillslope_selection_confirmed = True in the previous cell before continuing.')

if len(hillslope_endpoint_labels) != 2 or hillslope_endpoint_labels[0] == hillslope_endpoint_labels[1]:
    raise ValueError('Provide two different P labels in hillslope_endpoint_labels.')

mask_by_label = {}
for result in route_results:
    for mask in result['masks']:
        mask_by_label.setdefault(mask[0], mask)

missing_labels = [label for label in hillslope_endpoint_labels if label not in mask_by_label]
if missing_labels:
    raise ValueError(f'{missing_labels} have no calculated drainage boundary. Add them to selected_endpoint_pairs and rerun the route calculation.')

first_label, second_label = hillslope_endpoint_labels
first_mask, second_mask = mask_by_label[first_label], mask_by_label[second_label]
shared_boundary = first_mask[2] & second_mask[2]
exclusive_boundary = first_mask[2] ^ second_mask[2]
if not exclusive_boundary.any():
    raise ValueError(f'{first_label} and {second_label} have no non-shared drainage-boundary segment.')

# The two junctions are shared pixels immediately adjacent to the non-shared boundary segment.
pixel_neighborhood = np.ones((3, 3), dtype=bool)
junction_pixels = shared_boundary & ndimage.binary_dilation(exclusive_boundary, structure=pixel_neighborhood)
junction_labels, junction_count = ndimage.label(junction_pixels, structure=pixel_neighborhood)
if junction_count != 2:
    raise ValueError(
        f'Expected two non-shared-boundary junctions for {first_label}/{second_label}, found {junction_count}. '
        'Inspect the two drainage boundaries before selecting a different P-label pair.'
    )

def junction_pixel(component_index):
    rows, cols = np.where(junction_labels == component_index)
    x, y = affine * (cols, rows)
    center_index = np.argmin(np.hypot(x - x.mean(), y - y.mean()))
    return {'start_col': int(cols[center_index]), 'start_row': int(rows[center_index]),
            'start_x': float(x[center_index]), 'start_y': float(y[center_index])}

hillslope_start_points = pd.DataFrame([junction_pixel(1), junction_pixel(2)])
hillslope_start_points.index = [f'Hillslope {index}' for index in range(1, len(hillslope_start_points) + 1)]

site_matches = site_locations.loc[site_locations.site_id == hillslope_site_id]
if len(site_matches) != 1:
    raise ValueError(f'Expected one site named {hillslope_site_id}, found {len(site_matches)}.')
site = site_matches.iloc[0]
pour_x, pour_y = grid.snap_to_mask(stream_mask, (site.x_daymet, site.y_daymet))
pour_col, pour_row = grid.nearest_cell(pour_x, pour_y)
hillslope_pour_point = {'site_id': hillslope_site_id, 'pour_col': int(pour_col), 'pour_row': int(pour_row),
                         'pour_x': float(pour_x), 'pour_y': float(pour_y)}

display(hillslope_start_points[['start_col', 'start_row', 'start_x', 'start_y']])
display(pd.DataFrame([hillslope_pour_point]))


In [ ]:
# Visualize potential hillslope start points and transect lines

if not flag_hillslope_selection_confirmed:
    raise RuntimeError('Set flag_hillslope_selection_confirmed = True in the confirmation cell before continuing.')

fig, ax = plt.subplots(figsize=(12, 10))
watershed_workflow.plot.rivers(rivers, cache_crs, ax=ax, colors='gold', linewidth=0.7, zorder=2)
ax.plot([], [], color='gold', linewidth=0.7, label='NHD Plus river network')
ax.add_patch(MplPolygon(cell_corners, closed=True, fill=False, edgecolor='magenta', linewidth=2.5, linestyle='--', label=grid_id))
for site in site_locations.itertuples():
    ax.plot(site.x_daymet, site.y_daymet, 'k*', markersize=14, zorder=8, label=f'{site.site_id} input')
plot_masks = {}
for result in route_results:
    for mask in result['masks']:
        plot_masks.setdefault(mask[0], mask)
for color, (label, mask) in zip(plt.cm.tab10(np.linspace(0, 1, len(plot_masks))), plot_masks.items()):
    rows, cols = np.where(mask[2])
    x, y = affine * (cols, rows)
    ax.scatter(x, y, s=7, color=color, alpha=0.65, zorder=3, label=f'{label} drainage boundary')

ax.plot(hillslope_pour_point['pour_x'], hillslope_pour_point['pour_y'], 'ks', markersize=9,
        zorder=7, label=f"{hillslope_pour_point['site_id']} pour point")
for index, start_point in enumerate(hillslope_start_points.itertuples(), start=1):
    ax.plot([start_point.start_x, hillslope_pour_point['pour_x']], [start_point.start_y, hillslope_pour_point['pour_y']],
            color='black', linewidth=2.5, zorder=6, label=f'Hillslope {index}')
    ax.plot(start_point.start_x, start_point.start_y, '^', color='black', markersize=12,
            markeredgecolor='black', markeredgewidth=1.5, zorder=7)
    ax.annotate(f'Hillslope {index}',
                ((hillslope_pour_point['pour_x'] + start_point.start_x) / 2, (hillslope_pour_point['pour_y'] + start_point.start_y) / 2),
                xytext=(5, 5), textcoords='offset points', color='black', fontweight='bold')

ax.set(xlim=(min(xs) - margin, max(xs) + margin), ylim=(min(ys) - margin, max(ys) + margin),
       xlabel='Easting [m]', ylabel='Northing [m]',
       title=f'Potential hillslope start points: {first_label}/{second_label} to {hillslope_site_id}')
ax.legend(loc='best', fontsize=8)
ax.grid(True, alpha=0.25)
candidate_path = output_dir / f'{first_label}_{second_label}_potential_hillslope_start_points.png'
fig.savefig(candidate_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved {candidate_path}')


In [ ]:
# Hillslope transects with DEM background

if not flag_hillslope_selection_confirmed:
    raise RuntimeError('Set flag_hillslope_selection_confirmed = True in the confirmation cell before continuing.')

dem_extent = (dem_bounds[0], dem_bounds[2], dem_bounds[1], dem_bounds[3])
fig, ax = plt.subplots(figsize=(12, 10))
image = ax.imshow(dem, cmap='terrain', extent=dem_extent, origin='upper', zorder=0)
watershed_workflow.plot.rivers(rivers, cache_crs, ax=ax, colors='darkred', linewidth=0.7, zorder=2)
ax.plot([], [], color='darkred', linewidth=0.7, label='NHD Plus river network')
ax.add_patch(MplPolygon(cell_corners, closed=True, fill=False, edgecolor='magenta', linewidth=2.5, linestyle='--', label=grid_id))

ax.plot(hillslope_pour_point['pour_x'], hillslope_pour_point['pour_y'], 'ks', markersize=9,
        zorder=7, label=f"{hillslope_pour_point['site_id']} pour point")
# for index, start_point in enumerate(hillslope_start_points.itertuples(), start=1):
#     ax.plot([start_point.start_x, hillslope_pour_point['pour_x']], [start_point.start_y, hillslope_pour_point['pour_y']],
#             color='black', linewidth=2.5, zorder=6, label=f'Hillslope {index}')
#     ax.annotate(f'Hillslope {index}',
#                 ((hillslope_pour_point['pour_x'] + start_point.start_x) / 2,
#                  (hillslope_pour_point['pour_y'] + start_point.start_y) / 2),
#                 xytext=(5, 5), textcoords='offset points', color='black', fontweight='bold')
start_point = hillslope_start_points.iloc[0]

ax.plot([start_point.start_x, hillslope_pour_point['pour_x']],
        [start_point.start_y, hillslope_pour_point['pour_y']],
        color='black', linewidth=2.5, zorder=6, label='Hillslope 1')

ax.annotate('Hillslope 1',
            ((hillslope_pour_point['pour_x'] + start_point.start_x) / 2,
             (hillslope_pour_point['pour_y'] + start_point.start_y) / 2),
            xytext=(5, 5), textcoords='offset points',
            color='black', fontweight='bold')

ax.set(xlim=(min(xs) - margin, max(xs) + margin), ylim=(min(ys) - margin, max(ys) + margin),
       xlabel='Easting [m]', ylabel='Northing [m]',
       title=f'Hillslope transects: {first_label}/{second_label} to {hillslope_site_id}')
ax.legend(loc='best', fontsize=8)
ax.grid(True, alpha=0.25)
plt.colorbar(image, ax=ax, fraction=.046, pad=.04, label='Elevation [m]')
dem_candidate_path = output_dir / f'{first_label}_{second_label}_hillslope_transects_dem.png'
fig.savefig(dem_candidate_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved {dem_candidate_path}')


In [ ]:
# Summarize potential hillslope start, end, and length

if not flag_hillslope_selection_confirmed:
    raise RuntimeError('Set flag_hillslope_selection_confirmed = True in the confirmation cell before continuing.')

hillslope_summary = hillslope_start_points.copy()
hillslope_summary['end_site_id'] = hillslope_pour_point['site_id']
hillslope_summary['end_col'] = hillslope_pour_point['pour_col']
hillslope_summary['end_row'] = hillslope_pour_point['pour_row']
hillslope_summary['end_x'] = hillslope_pour_point['pour_x']
hillslope_summary['end_y'] = hillslope_pour_point['pour_y']
hillslope_summary['hillslope_length_m'] = np.hypot(
    hillslope_summary['end_x'] - hillslope_summary['start_x'],
    hillslope_summary['end_y'] - hillslope_summary['start_y'],
)

display(hillslope_summary[['start_col', 'start_row', 'start_x', 'start_y',
                           'end_site_id', 'end_col', 'end_row', 'end_x', 'end_y',
                           'hillslope_length_m']].round({'start_x': 2, 'start_y': 2,
                                                         'end_x': 2, 'end_y': 2,
                                                         'hillslope_length_m': 2}))


In [ ]:
# Export v4 hillslope candidates for the BC-head review notebook
from pathlib import Path

candidate_dir = Path('../data-processed') / site_name
candidate_dir.mkdir(parents=True, exist_ok=True)
candidate_csv = candidate_dir / f'hillslope_candidates_{site_name}_v4.csv'

hillslope_candidates = hillslope_summary.reset_index().rename(columns={'index': 'hillslope_id'}).copy()
hillslope_candidates['endpoint_label_a'] = first_label
hillslope_candidates['endpoint_label_b'] = second_label
hillslope_candidates['candidate_source'] = '0a-transect_latlon.Naches.v4.D8.ipynb'
hillslope_candidates.to_csv(candidate_csv, index=False)
display(hillslope_candidates)
print(f'Wrote {len(hillslope_candidates)} v4 hillslope candidates to {candidate_csv}')
